In [ ]:
#aggregation

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from tqdm import tqdm

# ==========================================
# 1. SETUP & PATHS (MULTI-PLATE)
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
          "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
          "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

# Paths for the two separate outputs
OUTPUT_CSV_MEDIAN = os.path.join(PROJECT_ROOT,"10marchecht", "aggregated_wells_median.csv")
OUTPUT_CSV_STD = os.path.join(PROJECT_ROOT,"10marchecht", "aggregated_wells_std.csv")

CELL_COUNT_THRESHOLD = 0  
TREATMENT_COL = "Treatment"

all_plates_median = []
all_plates_std = []
all_cell_counts = [] 

for plate_id in PLATES:
    print(f"\n--- Processing {plate_id} ---")
    
    FEATURES_BASE = os.path.join(PROJECT_ROOT, "features", plate_id)
    METADATA_PATH = os.path.join(PROJECT_ROOT, "metadata", f"index_{plate_id}.csv")
    
    if not os.path.exists(METADATA_PATH):
        print(f"Skipping {plate_id}: Metadata not found.")
        continue

    meta = pd.read_csv(METADATA_PATH)
    well_storage = {}
    well_to_treatment = {}

    for i in tqdm(meta.index, desc=f"Loading {plate_id}"):
        well_id = f"{plate_id}_{meta.loc[i, 'Metadata_Well']}"
        treatment = str(meta.loc[i, TREATMENT_COL]).strip()
        
        filename = os.path.join(FEATURES_BASE, 
                                str(meta.loc[i, "Metadata_Well"]), 
                                f"{meta.loc[i, 'Metadata_Site']}.npz")
        
        if os.path.isfile(filename):
            try:
                with np.load(filename) as data:
                    cells = data["features"]
                    cells_f = cells[~np.isnan(cells).any(axis=1)]
                    
                    if len(cells_f) > 0:
                        if well_id not in well_storage:
                            well_storage[well_id] = []
                            well_to_treatment[well_id] = treatment
                        well_storage[well_id].append(cells_f)
            except:
                continue

    # --- AGGREGATION & THRESHOLDING STEP ---
    for well_id, feature_list in well_storage.items():
        all_cells_in_well = np.vstack(feature_list)
        well_cell_count = all_cells_in_well.shape[0]
        all_cell_counts.append(well_cell_count)

        if well_cell_count >= CELL_COUNT_THRESHOLD:
            # Calculate both Median and Std Dev
            well_median = np.median(all_cells_in_well, axis=0)
            well_std = np.std(all_cells_in_well, axis=0)
            
            base_info = {
                "Plate": plate_id, 
                "Well_ID": well_id, 
                "Treatment": well_to_treatment[well_id],
                "Cell_Count": well_cell_count
            }
            
            # Create rows for both dataframes
            row_median = base_info.copy()
            row_std = base_info.copy()
            
            for idx in range(len(well_median)):
                row_median[idx] = well_median[idx]
                row_std[idx] = well_std[idx]
                
            all_plates_median.append(row_median)
            all_plates_std.append(row_std)

# Convert to DataFrames
df_median = pd.DataFrame(all_plates_median)
df_std = pd.DataFrame(all_plates_std)

# Helper function to reorder
def reorder_cols(df):
    meta_cols = ["Plate", "Well_ID", "Treatment", "Cell_Count"]
    feat_cols = [c for c in df.columns if c not in meta_cols]
    return df[meta_cols + feat_cols]

df_median = reorder_cols(df_median)
df_std = reorder_cols(df_std)

print(f"\nAggregation complete.")

# ==========================================
# 2. VISUALIZATION: CELL COUNT HISTOGRAM
# ==========================================
counts = np.array(all_cell_counts)
c_mean = np.mean(counts)
c_median = np.median(counts)
c_std = np.std(counts)

plt.figure(figsize=(10, 6))
plt.hist(counts, bins=50, color='skyblue', edgecolor='black', alpha=0.7)

# Create stats text string
stats_text = f'Mean: {c_mean:.2f}\nMedian: {c_median:.2f}\nStd Dev: {c_std:.2f}'
# Place text box in the plot
plt.gca().text(0.95, 0.95, stats_text, transform=plt.gca().transAxes, 
               verticalalignment='top', horizontalalignment='right',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

plt.title('Distribution of Cell Counts per Well (All Plates)')
plt.xlabel('Number of Cells')
plt.ylabel('Frequency (Wells)')
plt.grid(axis='y', alpha=0.3)
plt.show()

# ==========================================
# 3. SAVE TO CSVs
# ==========================================
df_median.to_csv(OUTPUT_CSV_MEDIAN, index=False)
df_std.to_csv(OUTPUT_CSV_STD, index=False)

print(f"Median data saved to: {OUTPUT_CSV_MEDIAN}")
print(f"Std Dev data saved to: {OUTPUT_CSV_STD}")

In [ ]:
#counting patches

In [ ]:
import numpy as np
import os
from tqdm import tqdm

PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
          "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
          "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

total_patches = 0
total_files = 0

print("Calculating total patch count...")

for plate in PLATES:
    feature_path = os.path.join(PROJECT_ROOT, "features", plate)
    
    if not os.path.exists(feature_path):
        print(f"Skipping {plate}: Path not found.")
        continue
        
    # Walk through all well subfolders
    for root, dirs, files in os.walk(feature_path):
        for file in files:
            if file.endswith(".npz"):
                file_path = os.path.join(root, file)
                try:
                    with np.load(file_path) as data:
                        # 'features' is the standard key in DeepProfiler npz files
                        # We only need the shape[0] (number of rows/cells)
                        total_patches += data["features"].shape[0]
                        total_files += 1
                except Exception as e:
                    print(f"Could not read {file}: {e}")

print("\n--- Final Statistics ---")
print(f"Total .npz files (sites) processed: {total_files}")
print(f"Total number of patches (cells):     {total_patches:,}")

In [ ]:
#removing mutants with less than 5 patches

In [ ]:
import pandas as pd
import os

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_CSV = os.path.join(PROJECT_ROOT, "10marchecht", "aggregated_wells_median.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "10marchecht")

# Create the directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. DEFINE YOUR NEW THRESHOLD
STRICT_THRESHOLD = 5 

# 3. LOAD & FILTER
print(f"Loading {INPUT_CSV}...")
df = pd.read_csv(INPUT_CSV)

# Identify the wells that ARE BELOW the threshold
removed_df = df[df['Cell_Count'] < STRICT_THRESHOLD].copy()

# Identify the wells that ARE ABOVE or EQUAL to the threshold
filtered_df = df[df['Cell_Count'] >= STRICT_THRESHOLD].copy()

# 4. REPORT & SAVE
print(f"\n--- Filtering Summary ---")
print(f"Original wells:       {len(df)}")
print(f"Wells kept:           {len(filtered_df)}")
print(f"Wells removed:        {len(removed_df)}")
print(f"-------------------------")

if not removed_df.empty:
    print(f"\n--- LIST OF REMOVED WELLS (Count < {STRICT_THRESHOLD}) ---")
    # We only show the metadata columns for the removed wells
    print(removed_df[['Plate', 'Well_ID', 'Treatment', 'Cell_Count']].to_string(index=False))
else:
    print("\nNo wells were below the threshold.")

# 5. SAVE DATA
OUTPUT_CSV_FILTERED = os.path.join(OUTPUT_DIR, f"aggregated_wells_median_min5.csv")
filtered_df.to_csv(OUTPUT_CSV_FILTERED, index=False)

print(f"\nDone! Filtered data saved to: {OUTPUT_CSV_FILTERED}")

In [ ]:
#feature seleciton

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_CSV = os.path.join(PROJECT_ROOT, "10marchecht", "aggregated_wells_median_min5.csv") 
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "10marchecht")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
df_raw.columns = [str(c) for c in df_raw.columns]

# Separate Metadata and Features
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
feature_cols = [c for c in df_raw.columns if c not in metadata_cols]

# ==========================================
# TOOLBOX: FUNCTIONS
# ==========================================

def filter_within_plate_consistency(df, features, top_n_to_keep=500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

def filter_across_plate_stability(df, features, top_n_to_keep=300):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    if not tp_batch_noises:
        return features 
        
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    
    # Selection based on batch noise ranking
    stable_features = total_batch_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

def filter_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

# ==========================================
# EXECUTION PIPELINE
# ==========================================

# --- STEP 0: GLOBAL VARIANCE FILTER ---
# Removes features that are flat/near-zero across the whole experiment first
print(f"\nRunning Step 0: Global Variance Filter (std > 0.01)...")
initial_std = df_raw[feature_cols].std()
active_features = initial_std[initial_std > 0.01].index.tolist()
print(f"Removed {len(feature_cols) - len(active_features)} low-variance features.")

# --- STEP 1: WITHIN-PLATE CONSISTENCY ---
print(f"Running Step 1: Within-Plate Consistency (Filtering to top 3000)...")
step1_features = filter_within_plate_consistency(df_raw, active_features, top_n_to_keep=3000)
df_step1 = df_raw[metadata_cols + step1_features]

# --- STEP 2: ACROSS-PLATE STABILITY ---
print(f"Running Step 2: Across-Plate Stability (Filtering to top 200)...")
step2_features = filter_across_plate_stability(df_step1, step1_features, top_n_to_keep=200)
df_step2 = df_step1[metadata_cols + step2_features]

# --- STEP 3: REDUNDANCY REMOVAL ---
print(f"Running Step 3: Redundancy Filter (Threshold 0.9)...")
final_feature_list = filter_redundancy(df_step2, step2_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final = df_step2[metadata_cols + final_feature_list]
output_path = os.path.join(OUTPUT_DIR, "vettedcellcounts_10march.csv")
df_final.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Original features: {len(feature_cols)}")
print(f"Active features (Step 0): {len(active_features)}")
print(f"Final feature count: {len(final_feature_list)}")
print(f"Saved to: {output_path}")
print("="*40)

In [ ]:
plotting

In [ ]:
#thuis:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script
file_path = os.path.join(PROJECT_ROOT,"10marchecht","vettedcellcounts_10march.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT,"10marchecht", "UMAP_10march")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# FIXED: We identify features by checking if the column name is purely numeric
# This excludes 'Plate', 'Well_ID', 'Treatment', 'Cell_Count', and 'Type'
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']} ({len(config['indices'])} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters - n_neighbors affects local vs global structure
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        # ADDED: 'Cell_Count' to hover_data so you can check quality during exploration
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=850, 
        height=850,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_path = os.path.join(OUTPUT_DIR, f"UMAP5_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()

In [ ]:
#nopreselect

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
file_path = os.path.join(PROJECT_ROOT, "10marchecht", "aggregated_wells_median_min5.csv")
df = pd.read_csv(file_path)

HTML_DIR = os.path.join(PROJECT_ROOT, "10marchecht", "UMAP_nopreprosess")
SVG_DIR = os.path.join(PROJECT_ROOT, "10marchecht", "UMAP_nopreprosessVector_Scalable")

for folder in [HTML_DIR, SVG_DIR]:
    if not os.path.exists(folder):
        os.makedirs(folder)

df.columns = [str(c) for c in df.columns]
df['Timepoint'] = df['Plate'].str.extract(r'(T\d+)')
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if str(x).lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# --- 2. COLOR SCHEMES ---
unique_combos = sorted(df['Plate'].unique())
turbo_colors = px.colors.sample_colorscale("Turbo", [i/(len(unique_combos)-1) for i in range(len(unique_combos))])
combo_color_map = {combo: turbo_colors[i] for i, combo in enumerate(unique_combos)}

time_colors = {'T0': '#A9D1FF', 'T1': '#2A7FFF', 'T2': "#1647AA"}

# --- 3. CHANNEL SELECTION ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. EXECUTION ---
for config in plot_configs:
    if not config['indices']: continue
    print(f"Generating Plots and SVGs for: {config['name']}...")
    
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]

    modes = [
        ('PLATE', unique_combos, combo_color_map, 'Plate-Time View'),
        ('TIME', sorted(df_plot['Timepoint'].unique()), time_colors, 'Timepoint View')
    ]

    for mode_name, groups, color_map, title_prefix in modes:
        fig = go.Figure()
        
        for group in groups:
            for t_type in ['Mutant', 'Control']:
                mask = (df_plot['Plate' if mode_name == 'PLATE' else 'Timepoint'] == group) & (df_plot['Type'] == t_type)
                curr = df_plot[mask]
                if curr.empty: continue
                
                color = color_map[group]
                
                fig.add_trace(go.Scatter(
                    x=curr['UMAP1'], y=curr['UMAP2'], mode='markers',
                    name=str(group),
                    marker=dict(
                        color=color, 
                        size=8 if t_type == 'Mutant' else 11,
                        symbol='circle' if t_type == 'Mutant' else 'square',
                        line=dict(width=1.0, color='black') if t_type == 'Control' else dict(width=0),
                        opacity=1.0
                    ),
                    customdata=np.stack((curr['Plate'], curr['Well_ID'], curr['Treatment'], curr['Cell_Count']), axis=-1),
                    hovertemplate="<b>%{customdata[2]}</b><br>Plate: %{customdata[0]}<br>Well: %{customdata[1]}<br>Count: %{customdata[3]}<extra></extra>",
                    showlegend=True if t_type == 'Mutant' else False,
                    legendgroup=str(group)
                ))

        # --- UPDATED LAYOUT SECTION ---
        fig.update_layout(
            title=f"{title_prefix}: {config['name']}",
            template='plotly_white',
            width=850, height=850,
            xaxis=dict(
                title="UMAP 1",
                showgrid=False,
                showticklabels=True,  # Show Axis Numbers
                showline=True,        # Show Axis Border
                linecolor='black',
                zeroline=False
            ),
            yaxis=dict(
                title="UMAP 2",
                showgrid=False,
                showticklabels=True,  # Show Axis Numbers
                showline=True,        # Show Axis Border
                linecolor='black',
                zeroline=False
            )
        )

        html_name = f"{config['name'].replace(' ', '_')}_{mode_name}.html"
        fig.write_html(os.path.join(HTML_DIR, html_name))
        
        svg_name = f"{config['name'].replace(' ', '_')}_{mode_name}.svg"
        fig.write_image(os.path.join(SVG_DIR, svg_name))

    print(f"Successfully saved {config['name']} exports.")

print("\nAll files (HTML and SVG) are ready.")

In [ ]:
#preprocessing

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'

file_path = os.path.join(PROJECT_ROOT,"10marchecht","vettedcellcounts_10march.csv")
df = pd.read_csv(file_path)


# Separate directories for HTML and SVG
HTML_DIR = os.path.join(PROJECT_ROOT, "10marchecht", "UMAP_preprosess")
SVG_DIR = os.path.join(PROJECT_ROOT, "10marchecht", "UMAP_preprosessVector_Scalable")

for folder in [HTML_DIR, SVG_DIR]:
    if not os.path.exists(folder):
        os.makedirs(folder)

df.columns = [str(c) for c in df.columns]

# Data Parsing
df['Timepoint'] = df['Plate'].str.extract(r'(T\d+)')
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if str(x).lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# --- 2. COLOR SCHEMES ---
unique_combos = sorted(df['Plate'].unique())
turbo_colors = px.colors.sample_colorscale("Turbo", [i/(len(unique_combos)-1) for i in range(len(unique_combos))])
combo_color_map = {combo: turbo_colors[i] for i, combo in enumerate(unique_combos)}

time_colors = {
    'T0': '#A9D1FF', 
    'T1': '#2A7FFF', 
    'T2': "#2058C9"  
}

# --- 3. CHANNEL SELECTION ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. EXECUTION ---
for config in plot_configs:
    if not config['indices']: continue
    print(f"Generating Plots and SVGs for: {config['name']}...")
    
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]

    # --- PLOT LOOP (Plate & Time) ---
    modes = [
        ('PLATE', unique_combos, combo_color_map, 'Plate-Time View'),
        ('TIME', sorted(df_plot['Timepoint'].unique()), time_colors, 'Timepoint View')
    ]

    for mode_name, groups, color_map, title_prefix in modes:
        fig = go.Figure()
        
        for group in groups:
            for t_type in ['Mutant', 'Control']:
                # Determine filtering logic based on mode
                mask = (df_plot['Plate' if mode_name == 'PLATE' else 'Timepoint'] == group) & (df_plot['Type'] == t_type)
                curr = df_plot[mask]
                if curr.empty: continue
                
                color = color_map[group]
                
                fig.add_trace(go.Scatter(
                    x=curr['UMAP1'], y=curr['UMAP2'], mode='markers',
                    name=str(group),
                    marker=dict(
                        color=color, 
                        size=8 if t_type == 'Mutant' else 11,
                        symbol='circle' if t_type == 'Mutant' else 'square',
                        line=dict(width=1.0, color='black') if t_type == 'Control' else dict(width=0),
                        opacity=1.0
                    ),
                    customdata=np.stack((curr['Plate'], curr['Well_ID'], curr['Treatment'], curr['Cell_Count']), axis=-1),
                    hovertemplate="<b>%{customdata[2]}</b><br>Plate: %{customdata[0]}<br>Well: %{customdata[1]}<br>Count: %{customdata[3]}<extra></extra>",
                    showlegend=True if t_type == 'Mutant' else False,
                    legendgroup=str(group)
                ))

        # --- UPDATED LAYOUT SECTION (TICKS & FRAME ADDED) ---
        fig.update_layout(
            title=f"{title_prefix}: {config['name']}",
            template='plotly_white',
            width=850, height=850,
            xaxis=dict(
                title="UMAP 1",
                showgrid=False,
                showticklabels=True,  # Numbers ON
                ticks="outside",      # Ticks visible outside
                tickcolor='black',    # Black tick marks
                showline=True,        # Axis line ON
                linecolor='black',    # Axis line color
                mirror=False,          # Border on all sides
                zeroline=False        # No heavy 0 line
            ),
            yaxis=dict(
                title="UMAP 2",
                showgrid=False,
                showticklabels=True,  # Numbers ON
                ticks="outside",      # Ticks visible outside
                tickcolor='black',    # Black tick marks
                showline=True,        # Axis line ON
                linecolor='black',    # Axis line color
                mirror=False,          # Border on all sides
                zeroline=False        # No heavy 0 line
            )
        )

        # 1. Save HTML
        html_name = f"{config['name'].replace(' ', '_')}_{mode_name}.html"
        fig.write_html(os.path.join(HTML_DIR, html_name))
        
        # 2. Save SVG (requires kaleido)
        svg_name = f"{config['name'].replace(' ', '_')}_{mode_name}.svg"
        fig.write_image(os.path.join(SVG_DIR, svg_name))

    print(f"Successfully saved {config['name']} exports.")

print("\nAll files (HTML and SVG) are ready in the output folders.")

In [ ]:
#withplateseleciotn

In [ ]:
#thuis:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script
file_path = os.path.join(PROJECT_ROOT,"10marchecht","vettedcellcounts_10march.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT,"10marchecht", "UMAP_T0")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE2_T0",
                   "PLATE3_T0","PLATE4_T0",
                   "PLATE5_T0"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# FIXED: We identify features by checking if the column name is purely numeric
# This excludes 'Plate', 'Well_ID', 'Treatment', 'Cell_Count', and 'Type'
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']} ({len(config['indices'])} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters - n_neighbors affects local vs global structure
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        # ADDED: 'Cell_Count' to hover_data so you can check quality during exploration
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=1100, 
        height=800,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_path = os.path.join(OUTPUT_DIR, f"UMAP5_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()

In [ ]:
#T0andT1 samen

In [ ]:
#thuis:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script
file_path = os.path.join(PROJECT_ROOT,"10marchecht","vettedcellcounts_10march.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT,"10marchecht", "UMAP_T0T1")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE2_T0",
                   "PLATE3_T0","PLATE4_T0",
                   "PLATE5_T0","PLATE1_T1","PLATE2_T1",
                   "PLATE3_T1","PLATE4_T1",
                   "PLATE5_T1"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# FIXED: We identify features by checking if the column name is purely numeric
# This excludes 'Plate', 'Well_ID', 'Treatment', 'Cell_Count', and 'Type'
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']} ({len(config['indices'])} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters - n_neighbors affects local vs global structure
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        # ADDED: 'Cell_Count' to hover_data so you can check quality during exploration
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=1100, 
        height=800,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_path = os.path.join(OUTPUT_DIR, f"UMAP5_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Changed folder name to reflect this is the "No Labels" version
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "10marchecht", "UMAP_5_annotated_T0_T1")
COORD_DIR = os.path.join(OUTPUT_DIR, "Coordinates")

for folder in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(folder):
        os.makedirs(folder)

file_path = os.path.join(PROJECT_ROOT, "7marchecht", "vettedcellcounts5_9march_reordered.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

print("Loading data...")
df = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df.columns = [str(c) for c in df.columns]

# --- 2. PLATE/TIME FILTERING ---
SELECTED_PLATES = ["PLATE1_T0", "PLATE2_T0", "PLATE3_T0", "PLATE4_T0", "PLATE5_T0","PLATE1_T1", "PLATE2_T1", "PLATE3_T1", "PLATE4_T1", "PLATE5_T1"] 

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# --- 3. MERGE & FALLBACK ANNOTATIONS ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].fillna(df['SubtiWiki Annotation 3']).fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

# Define the Display Category
is_control = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
df['Display_Category'] = df['Effective_Annotation']
df.loc[is_control, 'Display_Category'] = 'no_sgrna'
df.loc[(df['Timepoint'] == 'T0') & (~is_control), 'Display_Category'] = 'Baseline (T0)'

# --- 4. DYNAMIC COLOR MAPPING (GOLDEN ANGLE TURBO) ---
all_cats = sorted([c for c in df['Display_Category'].unique() if c not in ['no_sgrna', 'Baseline (T0)', 'Unknown/Other']])
num_cats = len(all_cats)

if num_cats > 0:
    base_palette = px.colors.sample_colorscale("Turbo", [i/255 for i in range(256)])
    golden_ratio_conjugate = 0.618033988749895
    h_values = [(i * golden_ratio_conjugate) % 1 for i in range(num_cats)]
    max_contrast_palette = [base_palette[int(h * 255)] for h in h_values]
    color_map = {cat: max_contrast_palette[i] for i, cat in enumerate(all_cats)}
else:
    color_map = {}

color_map['no_sgrna'] = '#EBEBEB'
color_map['Baseline (T0)'] = '#B0B0B0'
color_map['Unknown/Other'] = '#222222'

# --- 5. EXECUTION ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    # Save coordinates
    coord_filename = f"Coordinates_{config['name']}.csv"
    df[['Plate', 'Well_ID', 'Treatment', 'Display_Category', 'UMAP1', 'UMAP2']].to_csv(os.path.join(COORD_DIR, coord_filename), index=False)
    
    # --- 6. PLOTTING (MODIFIED FOR NO TEXT LABELS) ---
    df['Point_Shape'] = df['Timepoint'].map({"T0": "circle", "T1": "x", "T2": "circle"})
    df.loc[is_control, 'Point_Shape'] = 'square'

    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Point_Shape',
        # 'text' parameter removed to declutter the graph
        symbol_map={"circle": "circle", "x": "x", "square": "square"},
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'Effective_Annotation': True, 'UMAP1': False, 'UMAP2': False},
        title=f"Pathway Overlay (T2 Only): {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    seen_pathways = set()
    fig.for_each_trace(lambda t: (
        t.update(showlegend=False) if t.name.split(",")[0] in seen_pathways 
        else (seen_pathways.add(t.name.split(",")[0]), t.update(name=t.name.split(",")[0]))
    ))

    # mode changed to 'markers' only
    fig.update_traces(
        mode='markers', 
        marker=dict(opacity=1.0, line=dict(width=0.5, color='white'))
    )
    
    # Marker sizes kept large for visibility
    fig.update_traces(marker=dict(size=10), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=14), selector=dict(marker_symbol='x'))      
    fig.update_traces(marker=dict(size=18, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=1600, height=1000,
        legend_title_text='Pathway / Group',
        annotations=[
            dict(
                text="<b>Key:</b> Square = no_sgrna | Cross (x) = T1 | Dot (●) = T2/T0",
                showarrow=False, xref="paper", yref="paper",
                x=0.5, y=1.07, font=dict(size=14),
                bgcolor="white", bordercolor="black", borderwidth=1
            )
        ],
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=18)), 
            showline=True, linewidth=2, linecolor='black', 
            showgrid=False, zeroline=False
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=18)), 
            showline=True, linewidth=2, linecolor='black', 
            showgrid=False, zeroline=False
        )
    )
    
    file_base = f"UMAP_T2_NoLabels_{config['name']}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{file_base}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{file_base}.svg"))

print(f"Done. Graphs generated without mutant labels for a cleaner view.")

In [ ]:
#feature selection met normalizaton

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_CSV = os.path.join(PROJECT_ROOT, "10marchecht", "aggregated_wells_median_min5.csv") 
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "10marchecht")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
df_raw.columns = [str(c) for c in df_raw.columns]

metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
feature_cols = [c for c in df_raw.columns if c not in metadata_cols]

# ==========================================
# NEW STEP: NORMALIZE BASED ON T0 CONTROLS
# ==========================================
print("\nRunning Normalization based on T0 Controls...")

# 1. Identify Controls at T0
# This assumes your plates are named like "PLATE1_T0", etc.
t0_controls = df_raw[
    (df_raw['Treatment'] == CONTROL_LABEL) & 
    (df_raw['Plate'].str.endswith('T0'))
]

if t0_controls.empty:
    print("WARNING: No T0 controls found! Check your Plate names or Treatment labels.")
else:
    # 2. Calculate Median and MAD from T0 Controls for every feature
    t0_ctrl_medians = t0_controls[feature_cols].median()
    t0_ctrl_mads = (t0_controls[feature_cols] - t0_ctrl_medians).abs().median()
    
    # Avoid division by zero if a feature has 0 variation in controls
    t0_ctrl_mads = t0_ctrl_mads.replace(0, 1)

    # 3. Apply Robust Scaling: (Value - Median) / MAD
    # This makes the "average" T0 control = 0
    df_raw[feature_cols] = (df_raw[feature_cols] - t0_ctrl_medians) / t0_ctrl_mads
    print(f"Normalization complete. Features are now scaled relative to {len(t0_controls)} T0 control wells.")

# ==========================================
# TOOLBOX: FUNCTIONS (Same as yours)
# ==========================================

def filter_within_plate_consistency(df, features, top_n_to_keep=500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

def filter_across_plate_stability(df, features, top_n_to_keep=300):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    if not tp_batch_noises:
        return features 
        
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    stable_features = total_batch_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

def filter_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

# ==========================================
# EXECUTION PIPELINE
# ==========================================

# --- STEP 0: GLOBAL VARIANCE FILTER ---
print(f"\nRunning Step 0: Global Variance Filter (std > 0.01)...")
initial_std = df_raw[feature_cols].std()
active_features = initial_std[initial_std > 0.01].index.tolist()
print(f"Removed {len(feature_cols) - len(active_features)} low-variance features.")

# --- STEP 1: WITHIN-PLATE CONSISTENCY ---
print(f"Running Step 1: Within-Plate Consistency (Filtering to top 3000)...")
step1_features = filter_within_plate_consistency(df_raw, active_features, top_n_to_keep=3000)
df_step1 = df_raw[metadata_cols + step1_features]

# --- STEP 2: ACROSS-PLATE STABILITY ---
print(f"Running Step 2: Across-Plate Stability (Filtering to top 200)...")
step2_features = filter_across_plate_stability(df_step1, step1_features, top_n_to_keep=200)
df_step2 = df_step1[metadata_cols + step2_features]

# --- STEP 3: REDUNDANCY REMOVAL ---
print(f"Running Step 3: Redundancy Filter (Threshold 0.9)...")
final_feature_list = filter_redundancy(df_step2, step2_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final = df_step2[metadata_cols + final_feature_list]
output_path = os.path.join(OUTPUT_DIR, "vetted_normalized_T0_10marchTest.csv")
df_final.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Original features: {len(feature_cols)}")
print(f"Final feature count: {len(final_feature_list)}")
print(f"Saved to: {output_path}")
print("="*40)

In [ ]:
#thuis:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script
file_path = os.path.join(PROJECT_ROOT,"10marchecht","vetted_normalized_T0_10marchTest.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT,"10marchecht", "UMAP_standaredized")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE2_T0",
                   "PLATE3_T0","PLATE4_T0",
                   "PLATE5_T0","PLATE1_T1","PLATE2_T1",
                   "PLATE3_T1","PLATE4_T1",
                   "PLATE5_T1","PLATE1_T2","PLATE2_T2",
                   "PLATE3_T2","PLATE4_T2",
                   "PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# FIXED: We identify features by checking if the column name is purely numeric
# This excludes 'Plate', 'Well_ID', 'Treatment', 'Cell_Count', and 'Type'
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']} ({len(config['indices'])} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters - n_neighbors affects local vs global structure
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        # ADDED: 'Cell_Count' to hover_data so you can check quality during exploration
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=1100, 
        height=800,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_path = os.path.join(OUTPUT_DIR, f"UMAP5_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()